## Feature Engineering

#### Import packages

In [25]:
import pandas as pd
import numpy as np

#### Load cleaned data

In [26]:
data = pd.read_parquet(
    "../data/processed/train_transaction_cleaned.parquet", 
    engine="fastparquet"
)

#### Create a copy

In [27]:
features_data = data.copy()

#### Create time-based features
`TransactionsDT` is a time offset in seconds. We can use it to extract useful relative time features.

In [28]:
features_data["transaction_hour"] = (
    features_data["TransactionDT"] / 3600
) % 24

features_data["transaction_day"] = (
    features_data["TransactionID"] / (3600 * 24)
).astype(int)

features_data["transaction_week"] = (
    features_data["transaction_day"] / 7
).astype(int)

# Add a night transaction flag
features_data["is_night_transaction"] = (
    (features_data["transaction_hour"] < 6) |
    (features_data["transaction_hour"] > 22)
).astype(int)

#### Create amount-based features

In [29]:
features_data["transaction_amount_log"] = np.log1p(
    features_data["TransactionAmt"]
)

# Create rounded amount indicators
features_data["is_round_amount"]  = (
    features_data["transaction_amount_log"] % 1 == 0
).astype(int)

features_data["is_high_amount"] = (
    features_data["TransactionAmt"] > 
    features_data["TransactionAmt"].quantile(0.95)
).astype(int)

#### Create card-based frequency features

In [30]:
card_columns = ["card1", "card2", "card3", "card5"]

for column in card_columns:
    if column in features_data.columns:
        frequency_map = features_data[column].value_counts()
        features_data[f"{column}_frequency"] = features_data[column].map(
            frequency_map
        )

#### Create email-domain features

In [31]:
email_columns = [
    column for column in ["P_emaildomain", "R_emaildomain"]
    if column in features_data.columns
    ]

for column in email_columns:
    features_data[f"{column}_is_gmail"] = (
        features_data[column].str.contains("gmail", case=False, na=False)
    ).astype(int)

    features_data[f"{column}_is_yahoo"] = (
        features_data[column].str.contains("yahoo", case=False, na=False)
    ).astype(int)

    features_data[f"{column}_is_hotmail"] = (
        features_data[column].str.contains("hotmail", case=False, na=False)
    ).astype(int)

    features_data[f"{column}_is_outlook"] = (
        features_data[column].str.contains("outlook", case=False, na=False)
    ).astype(int)

#### Create address-match feature

In [32]:
if "addr1" in features_data.columns and "addr2" in features_data.columns:
    features_data["addr_match"] = (
        features_data["addr1"] == features_data["addr2"]
    ).astype(int)




#### Check new columns

In [33]:
new_columns = [
    column for column in features_data.columns
    if column not in data.columns
    ]

new_columns

['transaction_hour',
 'transaction_day',
 'transaction_week',
 'is_night_transaction',
 'transaction_amount_log',
 'is_round_amount',
 'is_high_amount',
 'card1_frequency',
 'card2_frequency',
 'card3_frequency',
 'card5_frequency',
 'P_emaildomain_is_gmail',
 'P_emaildomain_is_yahoo',
 'P_emaildomain_is_hotmail',
 'P_emaildomain_is_outlook',
 'R_emaildomain_is_gmail',
 'R_emaildomain_is_yahoo',
 'R_emaildomain_is_hotmail',
 'R_emaildomain_is_outlook',
 'addr_match']

#### Save feature-engineered dataset

In [34]:
features_data.to_parquet(
    "../data/processed/train_transaction_features.parquet",
    engine="fastparquet",
    index=False
)

## Feature Engineering Summary

Created additional transaction-level features to improve model performance beyond the raw dataset columns.

### New Features Added

- Extracted relative transaction hour, day, and week from `TransactionDT`
- Added a night-transaction indicator
- Created log-transformed transaction amount
- Added high-amount and round-amount indicators
- Added card-frequency features for selected card columns
- Added email-domain indicators for common providers
- Added address-match indicator when address fields were available

These engineered features are designed to help the model identify suspicious timing, unusual transaction amounts, repeated card patterns, and potentially meaningful account or identity signals.